In [18]:
import pandas as pd
import numpy as np
from datasets import Dataset
import torch

In [ ]:
!pip install -q -U transformers datasets peft sentence-transformers faiss-cpu wandb scikit-learn lightgbm torchao

In [14]:
train = pd.read_csv('/content/train.csv')
train_ds = Dataset.from_pandas(train)

# Milestone 2

In [16]:
def add_combined_text(example):
    return {
        "combined_text": f"{example['prompt']} {example['A']}"
    }

train_ds = train_ds.map(add_combined_text)

char_length = len(train_ds[51]["combined_text"])

print(char_length)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

614


In [21]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

sep_id = tokenizer.convert_tokens_to_ids("[SEP]")

encodings = tokenizer(
    list(train_ds["prompt"]),
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt",
)


inputs = tokenizer(train_ds[0]["prompt"], return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

cls_embedding = outputs.last_hidden_state[0, 0]

answer = cls_embedding[:5].sum().item()

print(round(answer, 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


-1.2001


In [22]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("bert-base-uncased")

head_size = config.hidden_size // config.num_attention_heads

print(head_size)

64


In [23]:
model = AutoModel.from_pretrained(
"bert-base-uncased", output_attentions=True)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "Light-ion fusion is a technique."

# Tokenize
inputs = tokenizer(text, return_tensors="pt")

# Forward pass
with torch.no_grad():
    outputs = model(**inputs)

# Get last layer attention: shape [batch, heads, seq_len, seq_len]
attentions = outputs.attentions[-1]

# First head
head0_attn = attentions[0, 0]

# Find token index for "fusion"
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(tokens)

fusion_idx = tokens.index("fusion")

# CLS token is always index 0
cls_idx = 0

# Attention from CLS -> fusion
weight = head0_attn[cls_idx, fusion_idx].item()

print(round(weight, 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
0.1025


In [24]:
from sentence_transformers import SentenceTransformer, util

# Load model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Get row 0 text
prompt = train_ds[0]["prompt"]
option_b = train_ds[0]["B"]

# Encode both texts
embeddings = model.encode([prompt, option_b], convert_to_tensor=True)

# Compute cosine similarity
score = util.cos_sim(embeddings[0], embeddings[1])

print(round(score.item(), 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

0.7658


In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()

all_text = []
for row in train_ds:
    all_text.append(row["prompt"])
    all_text.append(row["A"])
    all_text.append(row["B"])
    all_text.append(row["C"])
    all_text.append(row["D"])
    all_text.append(row["E"])

vectorizer.fit(all_text)

def get_tfidf_top3(row):
    prompt_vec = vectorizer.transform([row["prompt"]])

    options = [row["A"], row["B"], row["C"], row["D"], row["E"]]
    opt_vecs = vectorizer.transform(options)

    sims = cosine_similarity(prompt_vec, opt_vecs)[0]

    ranked = np.argsort(sims)[::-1]  # descending
    labels = ["A","B","C","D","E"]

    return [labels[i] for i in ranked[:3]]

def map_at3(pred, true):
    return sum([int(true == p) / (i+1) for i,p in enumerate(pred)])

tfidf_map3 = 0
tfidf_preds = []

for row in train_ds:
    preds = get_tfidf_top3(row)
    tfidf_preds.append(preds)
    tfidf_map3 += map_at3(preds, row["answer"])

tfidf_map3 /= len(train_ds)

print("TF-IDF MAP@3:", tfidf_map3)

TF-IDF MAP@3: 0.28491666666666604


In [26]:
def get_minilm_top3(row):
    prompt = row["prompt"]
    options = [row["A"], row["B"], row["C"], row["D"], row["E"]]

    emb = model.encode([prompt] + options, convert_to_tensor=True)

    prompt_emb = emb[0]
    opt_embs = emb[1:]

    sims = util.cos_sim(prompt_emb, opt_embs)[0]

    ranked = sims.argsort(descending=True)

    labels = ["A","B","C","D","E"]
    return [labels[i] for i in ranked[:3]]

minilm_map3 = 0
minilm_preds = []

for row in train_ds:
    preds = get_minilm_top3(row)
    minilm_preds.append(preds)

    true = row["answer"]
    minilm_map3 += sum([int(true == p) / (i+1) for i,p in enumerate(preds)])

minilm_map3 /= len(train_ds)

print("MiniLM MAP@3:", minilm_map3)

MiniLM MAP@3: 0.42308333333333487


In [27]:
count_improvement = 0

for i, row in enumerate(train_ds):
    true = row["answer"]

    tfidf_top3 = tfidf_preds[i]
    minilm_top3 = minilm_preds[i]

    if (true not in tfidf_top3) and (true in minilm_top3):
        count_improvement += 1

print("Improvement count:", count_improvement)

Improvement count: 570


In [28]:
from transformers import pipeline

# Initialize zero-shot pipeline (defaults to facebook/bart-large-mnli)
classifier = pipeline("zero-shot-classification")

# Get row 1 (index 1)
row = train_ds[1]

prompt = row["prompt"]
candidate_labels = [row["A"], row["B"], row["C"]]

# Run classification
result = classifier(prompt, candidate_labels)

# Get top probability score
top_score = result["scores"][0]

print(round(top_score, 4))

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

0.4575


In [29]:
row = train_ds[1]
prompt = row["prompt"]
labels = [row["A"], row["B"], row["C"]]

# --- Case 1: default (softmax)
res_softmax = classifier(prompt, labels, multi_label=False)
sum_softmax = sum(res_softmax["scores"])

# --- Case 2: multi_label=True (sigmoid)
res_sigmoid = classifier(prompt, labels, multi_label=True)
sum_sigmoid = sum(res_sigmoid["scores"])

# Absolute difference
diff = abs(sum_softmax - sum_sigmoid)

print(round(diff, 6))

0.99949


In [30]:
row = train_ds[0]

prompt_text = (
    f"Question: {row['prompt']}. "
    f"Is the correct answer A: {row['A']} or B: {row['B']}? "
    f"Answer with just the letter A or B."
)

# WORKING PIPELINE IN YOUR ENV
generator = pipeline(
    "text-generation",
    model="google/flan-t5-small"
)

output = generator(prompt_text, max_new_tokens=5)

print(output[0]["generated_text"])

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  308MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


...

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCaus

Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B.
